In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章只读几个模型的 config.json（各几 KB）并画几张小图，全程 CPU 秒级完成，
# 不下载任何模型权重、不做任何前向 / 反向。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来。
# transformers: 用 AutoConfig 读 config；GPT-2 是老模型不挑版本，Qwen3 要求 >=4.51，
#               故统一锁 >=4.51 一次装好。matplotlib / torch: Colab 默认已装。
!pip install -q -U "transformers>=4.51"

In [ ]:
# ============================================================
# Cell 2: 读 GPT-2 config，看早期 decoder-only 的配置
# ============================================================
from transformers import AutoConfig

# gpt2 = GPT-2 小号（124M）。AutoConfig 只下载 config.json（几 KB），不下载权重。
gpt2 = AutoConfig.from_pretrained("gpt2")

print("GPT-2 (124M) 关键配置：")
print(f"  层数         n_layer            = {gpt2.n_layer}")
print(f"  隐藏维度     n_embd             = {gpt2.n_embd}")
print(f"  注意力头数   n_head             = {gpt2.n_head}")
print(f"  上下文长度   n_positions        = {gpt2.n_positions}")
print(f"  词表大小     vocab_size         = {gpt2.vocab_size}")
print(f"  激活函数     activation_function= {gpt2.activation_function}")  # gelu_new
# GPT-2 的位置编码是 learned 绝对：config 里体现为有一张 n_positions 长的位置 embedding 表
# （wpe），而不像现代模型那样带 rope_theta。下面这句确认它【没有】rope 相关字段。
print(f"  有 rope_theta 吗？             = {hasattr(gpt2, 'rope_theta')}  (False => 用 learned 绝对位置，不是 RoPE)")
print(f"  归一化 eps   layer_norm_epsilon = {gpt2.layer_norm_epsilon}  (LayerNorm，不是 RMSNorm)")

In [ ]:
# ============================================================
# Cell 3: 读 Qwen3-8B config，逐项对上第 6 节的现代改动
# ============================================================
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-8B")

# rope_theta 在 config 里的位置随 transformers 版本而变：有的版本放在顶层 cfg.rope_theta，
# 有的挪进嵌套字典 cfg.rope_parameters["rope_theta"]。下面两处都试，兼容新旧版本。
rope_theta = getattr(cfg, "rope_theta", None)
if rope_theta is None:
    rope_theta = (getattr(cfg, "rope_parameters", None) or {}).get("rope_theta")

print("Qwen3-8B 关键配置（对上第 6 节现代改动清单）：")
print(f"  层数           num_hidden_layers   = {cfg.num_hidden_layers}")
print(f"  隐藏维度       hidden_size         = {cfg.hidden_size}")
print(f"  FFN 中间维度   intermediate_size   = {cfg.intermediate_size}")
print(f"  query 头数     num_attention_heads = {cfg.num_attention_heads}")
print(f"  KV 头数        num_key_value_heads = {cfg.num_key_value_heads}")
print(f"  RoPE base      rope_theta          = {rope_theta}")
print(f"  归一化 eps     rms_norm_eps        = {cfg.rms_norm_eps}")
print(f"  激活函数       hidden_act          = {cfg.hidden_act}")
print(f"  词表大小       vocab_size          = {cfg.vocab_size}")
print(f"  权重共享       tie_word_embeddings = {cfg.tie_word_embeddings}")

print("\n逐项对上第 6 节：")
print(f"  6.1 RoPE   ：有 rope_theta={rope_theta}      => 用 RoPE（非绝对位置）")
print(f"  6.2 RMSNorm：有 rms_norm_eps={cfg.rms_norm_eps} => 用 RMSNorm（非 LayerNorm）")
print(f"  6.3 SwiGLU ：hidden_act={cfg.hidden_act}        => SiLU 门控，即 SwiGLU（非 ReLU/GELU）")
print(f"  6.4 GQA    ：{cfg.num_attention_heads} query 头 > {cfg.num_key_value_heads} KV 头 => GQA（非 MHA）")
print(f"  6.5 MoE    ：Qwen3-8B 是稠密版、无 MoE 字段，故此处不列（MoE 版才有 num_experts 等字段）")
print(f"  6.6 tying  ：tie_word_embeddings={cfg.tie_word_embeddings}  => 8B 是大号，默认不共享")

In [ ]:
# ============================================================
# Cell 4: 可视化 GPT-1/2/3 的参数量增长（对数纵轴）
# ============================================================
import matplotlib.pyplot as plt

models = ["GPT-1\n(2018)", "GPT-2\n(2019)", "GPT-3\n(2020)"]
params_b = [0.117, 1.5, 175.0]   # 单位：十亿（B）参数：117M / 1.5B / 175B

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(models, params_b, color=["#93c5fd", "#60a5fa", "#2563eb"])
ax.set_yscale("log")             # 参数量跨 3 个数量级，线性轴会把前两根压成 0，必须用对数轴
ax.set_ylabel("Parameters (Billions, log scale)")
ax.set_title("GPT-1 -> GPT-2 -> GPT-3: ~1500x parameter growth in 2 years")
# 在每根柱子顶上标注具体数值
for b, p in zip(bars, params_b):
    label = f"{p*1000:.0f}M" if p < 1 else f"{p:.1f}B"
    ax.text(b.get_x() + b.get_width() / 2, p * 1.15, label, ha="center", va="bottom")
plt.tight_layout()
plt.show()

# 观察：纵轴每一格是 10 倍。GPT-1(117M) -> GPT-2(1.5B) 约 13 倍，
# GPT-2 -> GPT-3(175B) 约 117 倍，两年累计约 1500 倍——而架构几乎没变。
# 这张图就是「规模是 GPT 系列主旋律」最直白的证据。

In [ ]:
# ============================================================
# Cell 5: 打印「原版 / GPT-2 / LLaMA / Qwen3」零件对照矩阵
# ============================================================
# 每行：(零件, 原版Transformer, GPT-2, LLaMA, Qwen3)。星号 * 标记「这一代首次换上」。
rows = [
    ("整体架构",   "enc-dec 两栈",  "decoder-only",  "decoder-only",  "decoder-only"),
    ("归一化位置", "Post-LN",       "Pre-LN*",       "Pre-LN",        "Pre-LN"),
    ("归一化类型", "LayerNorm",     "LayerNorm",     "RMSNorm*",      "RMSNorm"),
    ("FFN 激活",   "ReLU",          "GELU",          "SwiGLU*",       "SwiGLU"),
    ("位置编码",   "sinusoidal",    "learned 绝对",  "RoPE*",         "RoPE"),
    ("注意力",     "MHA",           "MHA",           "GQA*(2代)",     "GQA"),
    ("线性层 bias","有",            "有",            "无*",           "无"),
    ("QK-norm",    "无",            "无",            "无",            "有*"),
    ("FFN 变体",   "稠密",          "稠密",          "稠密",          "稠密/MoE*"),
]
header = ("零件", "原版(2017)", "GPT-2(2019)", "LLaMA(2023)", "Qwen3(2025)")

# 计算每列宽度做对齐（中文按 2 个显示宽度粗略处理）
def w(s):  # 显示宽度：非 ASCII 记 2、ASCII 记 1
    return sum(2 if ord(c) > 127 else 1 for c in s)
cols = list(zip(header, *rows))
widths = [max(w(str(x)) for x in col) for col in cols]

def fmt_row(cells):
    return " | ".join(str(c) + " " * (widths[i] - w(str(c))) for i, c in enumerate(cells))

print(fmt_row(header))
print("-" * (sum(widths) + 3 * len(widths)))
for r in rows:
    print(fmt_row(r))
print("\n带 * 的是「该零件在这一代首次换上」。decoder-only、GELU、learned 绝对位置都始于 GPT-1（表中未单列），")
print("故 GPT-2 列这三项不带 *；由 GPT-2 首次立起的是 Pre-LN。顺着 * 往右读：GPT-2 立起 Pre-LN，")
print("LLaMA 换上 RMSNorm/SwiGLU/RoPE/GQA/去bias，Qwen3 再补 QK-norm 与可选 MoE。骨架自 2017 未变。")